In [52]:
from glob import glob
import os
from spectral.io import envi
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import h5py
from osgeo import osr
import pyproj

import rasterio

import sys
sys.path.append('/store/carroll/repos/SpectralUtil/spectral_util/')
import spec_io
import mosaic

%matplotlib widget

os.chdir('/store/carroll/col/data/')

In [12]:
obs = glob('2025/raw/L1/radianceENVI/*ALMO*OBS_Data')
obs = [x for x in obs if not any(L in x for L in ('L066', 'L067'))]
fp_out = '/store/carroll/repos/chess-isofit/2025/3c/3_mosaic/top_priority_obs_ALMO.txt'
with open(fp_out, 'w') as f:
    f.writelines(f'{x}\n' for x in obs)

In [13]:
obs = glob('2025/raw/L1/radianceENVI/*CRBU*OBS_Data')
obs = [x for x in obs if not any(L in x for L in ('L066', 'L067'))]
fp_out = '/store/carroll/repos/chess-isofit/2025/3c/3_mosaic/top_priority_obs_CRBU.txt'
with open(fp_out, 'w') as f:
    f.writelines(f'{x}\n' for x in obs)

In [14]:
obs = glob('2025/raw/L1/radianceENVI/*UPTA*OBS_Data')
obs = [x for x in obs if not any(L in x for L in ('L066', 'L067'))]
fp_out = '/store/carroll/repos/chess-isofit/2025/3c/3_mosaic/top_priority_obs_UPTA.txt'
with open(fp_out, 'w') as f:
    f.writelines(f'{x}\n' for x in obs)

In [10]:
sorted(obs)

['2025/raw/L1/radianceENVI/NIS01_20250613_161028_ALMO_DP1_L047-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_162441_ALMO_DP1_L048-3_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_163245_ALMO_DP1_L049-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_164033_ALMO_DP1_L050-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_164856_ALMO_DP1_L051-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_165657_ALMO_DP1_L052-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_170457_ALMO_DP1_L053-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_171225_ALMO_DP1_L054-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_171934_ALMO_DP1_L055-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_172653_ALMO_DP1_L056-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_173358_ALMO_DP1_L057-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_174100_ALMO_DP1_L058-1_OBS_Data',
 '2025/raw/L1/radianceENVI/NIS01_20250613_174833_ALMO_DP1_L059-1_OBS_Data',
 '2025/raw/L

In [136]:
# get the date and centroid of each sampling event
field = pd.read_csv('2025/raw/chess_site_form_2025_12_19.csv')
field = field[field['Sampling Area']!='Ad Hoc Site']

sampling_dates = (
    field
    .groupby(['Sampling Area', 'Collection Date'])
    .size()
    .reset_index(name='n')
)
sampling_dates = (
    sampling_dates
    .sort_values(['Sampling Area', 'n'], ascending=[True, False])
    .groupby('Sampling Area', as_index=False)
    .head(1)
    .reset_index(drop=True)
)
sampling_dates.loc[:,'Sampling Area'] = [x[:-5] for x in sampling_dates['Sampling Area']]

# estimate sampling_area centroids
field = gpd.read_file('2025/raw/chess_field_poly.geojson')
field = field.dissolve(by="Sampling_Area", as_index=False).to_crs(32613)
centroids = field.copy()
centroids["geometry"] = field.geometry.centroid
centroids = centroids[['Sampling_Area','geometry']]

# join

sampling_areas = centroids.merge(
    sampling_dates,
    left_on='Sampling_Area',
    right_on='Sampling Area',
    how='inner'
)
sampling_areas.to_file('2025/raw/sampling_areas.geojson')

In [138]:
# get cloud cover per fid

In [29]:
fps = glob('2025/raw/L1/radianceH5/*/*/*_radiance.h5')

In [32]:
fids = []
domains = []
dates = []
cloud_covers = []
cloud_types = []

for fp in fps:
    f = h5py.File(fp)
    fid = fp.split('/')[-1].removesuffix('_radiance.h5')
    domain = fid.split('_')[2]
    date = fid.split('_')[-1]
    cloud_cover = f[domain]['Radiance']['RadianceIntegerPart'].attrs['Cloud conditions']
    cloud_type = f[domain]['Radiance']['RadianceIntegerPart'].attrs['Cloud type']

    fids.append(fid)
    domains.append(domain)
    dates.append(date)
    cloud_covers.append(cloud_cover)
    cloud_types.append(cloud_type)

cloud_df = pd.DataFrame({'fid':fids, 'domain':domains, 'cloud_cover':cloud_covers, 'cloud_type':cloud_types})
cloud_df.to_csv('2025/raw/fid_clouds.csv')